# Fast S100 pure tangent-replay acquisition

This notebook acquires replay-complete physical trajectories for the later tangent-cocycle calculation. It saves the prepared and final occupied frames plus the complete ordered measurement record. No tangent frame, Choi covariance, covariance history, or Jacobian is computed here.

## 1. Editable configuration

The scientific grid and batch boundaries are locked. Use `REPORT_ONLY` to inspect resume state and `MAX_NEW_TASKS` to limit work in one Colab session.

In [ ]:
from pathlib import Path

BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/09_pure_tangent_replay_acquisition')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/pure_tangent_replay_nx20_ny24-32_hard-soft_a1-1-3_s100_v1')
SCRATCH_ROOT = Path('/content/pure_tangent_replay_acquisition_scratch')
LOCAL_BUNDLE_DIR = Path('/content/09_pure_tangent_replay_acquisition')
REPORT_ONLY = False
MAX_NEW_TASKS = None  # Set an integer to cap new batches in this session.

CONFIG = {
    'sampling_revision': 'pure_tangent_replay_nx20_ny24-32_hard-soft_a1-1-3_s100_v1',
    'root_seed': 2026090409,
    'Nx': 20,
    'Ny_values': [24, 28, 32],
    'alpha_1_values': [1.0, 3.0],
    'alpha_2': 30.0,
    'nshell': 1,
    'samples_per_case': 100,
    'batch_size_by_Ny': {'24': 50, '28': 50, '32': 25},
    'cycles_multiplier': 2,
    'trial_orbitals': 'X',
    'filling_fraction': 0.5,
    'n_a': 0.5,
    'init_mode': 'default',
    'sequence': 'raster_y',
    'perfect_correction': True,
    'postselect': False,
    'postselect_probability': 0.0,
    'dtype': 'complex128',
    'device': 'cuda:0',
    'backend': 'local',
    'state_representation': 'physical_frame',
    'triv_region_local_mode': False,
    'frame_reorthonormalize_interval': 1,
    'max_batch_runtime_seconds': 3600.0,
    'a100_peak_memory_limit_gib': 34.0,
    'constructions': {
        'hard': {'DW': True, 'dw_truncation': True, 'meas_slab_only': True},
        'soft': {'DW': True, 'dw_truncation': False, 'meas_slab_only': False},
    },
    'saved_products': {
        'prepared_initial_frame': True,
        'final_frame': True,
        'ordered_schedule': True,
        'bitpacked_outcomes': True,
        'bitpacked_targets': True,
        'cycle_log_probability': True,
        'intermediate_frames': False,
        'tangent_frame': False,
        'choi_covariance': False,
        'covariance_history': False,
    },
}
print(CONFIG)

## 2. Mount Drive and validate the runtime

Production requires an NVIDIA A100 with 40-GB-class memory. Report-only resume inspection may run without CUDA.

In [ ]:
import torch
from google.colab import drive

drive.mount('/content/drive')
if not REPORT_ONLY:
    if not torch.cuda.is_available():
        raise RuntimeError('Select an A100 GPU runtime before launching production.')
    props = torch.cuda.get_device_properties(0)
    if 'A100' not in props.name.upper() or props.total_memory < 38 * 1024**3:
        raise RuntimeError(f'Expected A100 40-GB class, found {props.name} with {props.total_memory / 1024**3:.2f} GiB')
    print(f'[runtime] {props.name}, {props.total_memory / 1024**3:.2f} GiB, complex128')
else:
    print('[runtime] report-only mode; CUDA launch is disabled')

## 3. Stage locally and run/resume

The bundle is copied to `/content`. The runner verifies Drive result/completion pairs, displays an outer batch `tqdm` bar, and surfaces the canonical engine's cycle progress.

In [ ]:
import codecs
import json
import os
import shutil
import subprocess
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/pure_tangent_replay_acquisition_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [
    sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
    '--config', str(config_path),
    '--output-root', str(OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_TASKS is not None:
    command.extend(['--max-new-tasks', str(int(MAX_NEW_TASKS))])
print('[launch]', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
if process.stdout is None:
    raise RuntimeError('Runner output pipe was not created')
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
try:
    while True:
        raw = os.read(process.stdout.fileno(), 4096)
        if not raw:
            break
        sys.stdout.write(decoder.decode(raw))
        sys.stdout.flush()
    tail = decoder.decode(b'', final=True)
    if tail:
        sys.stdout.write(tail)
        sys.stdout.flush()
except BaseException:
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
finally:
    process.stdout.close()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
print('[notebook] runner exited successfully', flush=True)

## 4. Release the runtime

Run this final cell after the queue or requested task limit completes.

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')